# LLMOps: RAG & Vector Databases (ChromaDB)

In [ ]:
# Day 47 — RAG & Vector Databases
#
# Problem: LLMs (like GPT-4) only know public information up until their training cutoff.
# If you ask: "How do I restart our company's internal billing API?", it will hallucinate
# or say "I don't know."
#
# Solution: RAG (Retrieval-Augmented Generation).
# 1. You put all your company wikis into a database.
# 2. When the user asks a question, you SEARCH the database for the relevant wiki page.
# 3. You paste that wiki page into the LLM's prompt and say "Answer using THIS context."
#
# To do this search efficiently, we don't use SQL. We use a Vector Database (like ChromaDB).
# Vector DBs search by *meaning* (semantic search) rather than exact keyword matches.
#
# Today we'll cover:
#   1. Document Loaders & Splitters
#   2. Embeddings (Turning text into numbers)
#   3. Storing in ChromaDB
#   4. The RAG Chain
#
# Install: pip install chromadb langchain-community tiktoken

### 1. Document Loaders and Splitters

In [ ]:
# First, let's create a fake company wiki.
import os
os.makedirs("wiki", exist_ok=True)
with open("wiki/billing_api.txt", "w") as f:
    f.write("The internal billing API is hosted on AKS cluster 'prod-east'. "
            "To restart it, run: kubectl rollout restart deployment billing-api -n finance.\n"
            "Escalation contact is Sarah on Slack.")

# Now we use LangChain to load the documents and split them into chunks.
# (LLMs have context limits, so we can't pass a 10,000 page PDF all at once).

rag_load_code = '''from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# 1. Load the documents
loader = DirectoryLoader('./wiki', glob="**/*.txt", loader_cls=TextLoader)
documents = loader.load()

# 2. Split them into chunks (e.g., 500 characters per chunk)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)

print(f"Loaded {len(documents)} documents, split into {len(chunks)} chunks.")
'''
print("Loading & Splitting Pattern:")
print(rag_load_code)

### 2. Embeddings and Vector Databases (ChromaDB)

In [ ]:
# Embeddings convert words into vectors (arrays of numbers) representing their meaning.
# "Dog" and "Puppy" have very similar vectors.
# We store these vectors in a Vector DB.

rag_embed_code = '''from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

# 1. Create the Embedding Model (OpenAI provides this via API)
embeddings = OpenAIEmbeddings()

# 2. Store the chunks in ChromaDB (a local Vector DB)
# This takes our chunks, converts them to vectors, and saves them to the './chroma_db' folder.
vector_db = Chroma.from_documents(
    documents=chunks, 
    embedding=embeddings, 
    persist_directory="./chroma_db"
)
'''
print("Embedding and Storing Pattern:")
print(rag_embed_code)

### 3. The RAG Chain

In [ ]:
# Now the magic happens. A user asks a question.
# We convert their question into a vector, find the most similar chunks in ChromaDB,
# and feed those chunks to the LLM.

rag_query_code = '''from langchain_openai import ChatOpenAI
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(temperature=0)

# 1. Create a Retriever (the search engine)
retriever = vector_db.as_retriever(search_kwargs={"k": 3}) # Get top 3 most relevant chunks

# 2. Define the RAG Prompt
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, say that you don't know.\n\n"
    "{context}"
)
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# 3. Build the Chains
# 'create_stuff_documents_chain' STUFFS the retrieved documents into the {context} variable.
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

# 4. Ask the Question!
response = rag_chain.invoke({"input": "How do I restart the billing API?"})
print(response["answer"])
'''
print("The RAG Retrieval Pattern:")
print(rag_query_code)

In [ ]:
# Practice Questions:
#
# 1. Why do we split large documents into "chunks" before storing them in a Vector DB?
# 2. What is the difference between a traditional SQL search (`LIKE %billing%`) and 
#    Semantic Search in a Vector DB?
# 3. In the RAG pattern, what exactly is the LLM analyzing? Is it searching the DB itself?

### DevOps / AI-MLOps Practice

In [ ]:
# Exercise 1: The SRE Runbook Assistant
# Your team has hundreds of Markdown files in a GitHub repo called "Runbooks" 
# (guides on what to do when specific services crash).
# 
# Describe the architecture required to build a Slack bot that SREs can ask:
# "The redis-cache cluster is throwing OOM errors, what is the mitigation?"
# (Mention DocumentLoaders, ChromaDB, OpenAIEmbeddings, and the RAG Chain).

# Why this matters for me as a DevOps/MLOps engineer:
# 1. Knowledge Silos: RAG destroys knowledge silos. Anyone can query the entire infrastructure wiki instantly.
# 2. No Fine-Tuning: Fine-tuning a model on company data costs thousands of dollars and takes weeks. 
#    RAG requires ZERO training, costs pennies, and updates instantly when a wiki page changes.